In [5]:
import pandas as pd
import sqlalchemy as sal

engine = sal.create_engine('mssql://ANKIT\SQLEXPRESS/master?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn=engine.connect()

In [28]:
def extract():
    df_orders = pd.read_csv('orders.txt')
    df_returns = pd.read_csv('returns.txt')
    return df_orders , df_returns

def transform(df_orders,df_returns):
    df = pd.merge(df_orders,df_returns, how = 'inner' , left_on='order_id' , right_on= 'order_id')
    return df

def load(df):
    df.to_sql('orders_final',con=conn , index=False , if_exists = 'append')
    conn.commit()

In [29]:
#extract
df_orders , df_returns = extract()

#tranform
df = transform(df_orders,df_returns)

#load
load(df)

In [30]:
df_sql = pd.read_sql_query('select * from orders_final' , conn)

In [31]:
df_sql

,order_id,order_date,customer_name,city,category,product_id,sales,profit,return_reason
0,CA-2018-100006,2018-09-07,Dennis Kane,New York City,Technology,TEC-PH-10002075,377.970,109.6113,bad quality
1,CA-2018-100090,2018-07-08,Ed Braxton,San Francisco,Furniture,FUR-TA-10003715,502.488,-87.9354,wrong product
2,CA-2018-100293,2018-03-14,Neil Französisch,Jacksonville,Office Supplies,OFF-PA-10000176,91.056,31.8696,bad quality
3,CA-2018-100328,2018-01-28,Jasper Cacioppo,New York City,Office Supplies,OFF-BI-10000343,3.928,1.3257,bad quality
4,CA-2018-100363,2018-04-08,Jim Mitchum,Glendale,Office Supplies,OFF-FA-10000611,2.368,0.8288,wrong product
5,CA-2018-100391,2018-05-25,Barry Weirich,New York City,Office Supplies,OFF-PA-10001471,14.620,6.7252,wrong product
6,CA-2018-100678,2018-04-18,Kunst Miller,Houston,Furniture,FUR-CH-10002602,317.058,-18.1176,others
7,CA-2018-100706,2018-12-16,Laurel Elliston,Springfield,Furniture,FUR-FU-10002268,29.460,9.7218,others
8,CA-2018-100762,2018-11-24,Nat Gilpin,Jackson,Office Supplies,OFF-AR-10000380,151.920,45.5760,bad quality
9,CA-2018-100860,2018-03-26,Cindy Stewart,Pomona,Office Supplies,OFF-LA-10001982,18.750,9.0000,wrong product


In [83]:
def extract():
    df_products = pd.read_csv('products.txt')
    df_products_db = pd.read_sql_query('select * from products' , conn)
    return df_products,df_products_db

def transform(df_products,df_products_db):
    df_merged = pd.merge(df_products , df_products_db , how='left' , on = 'product_id')
    df_insert = df_merged[df_merged['product_name_y'].isna()]
    df_insert_final =  df_insert.iloc[: , 0:3]
    df_insert_final.columns = df_products_db.columns
    
    df_update = df_merged[df_merged['product_name_y'].notna()]
    df_update_final =  df_update.iloc[: , 0:3]
    df_update_final.columns = df_products_db.columns
    return df_insert_final,df_update_final

def load_staging(df_update_final):
    df_update_final.to_sql('products_stg',con=conn , index=False , if_exists = 'replace')
    conn.commit()

def updates():
    query = sal.text("update products set price = products_stg.price, product_name=products_stg.product_name from products_stg where products.product_id=products_stg.product_id")
    p = conn.execute(query)
    conn.commit()
    
def inserts(df_insert_final):
    df_insert_final.to_sql('products',con=conn , index=False , if_exists = 'append')
    conn.commit()

In [84]:
df_products,df_products_db = extract()

df_insert_final,df_update_final = transform(df_products,df_products_db)

load_staging(df_update_final)

inserts(df_insert_final)

updates()